# 02. Indexación vectorial

Se construye el índice con los artículos admitidos por la ingesta. La configuración controla el encoder y las ventanas. La búsqueda devuelve unidades completas.

Este notebook todavía no se ha ejecutado. Usar el kernel AI Week. La segunda construcción sirve para comprobar reproducibilidad y vuelve a procesar todo el corpus.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

raiz = Path.cwd()
if not (raiz / "configs").exists():
    raiz = raiz.parent
config = json.loads((raiz / "configs/indice.json").read_text(encoding="utf-8"))
print(config["encoder"]["repo_id"])
print("Tokens:", config["tamano_tokens"], "Solapamiento:", config["solapamiento_tokens"])
(raiz / "reports").mkdir(exist_ok=True)

## Construcción

Se usa E5-large con 320 tokens y solapamiento de 32. Quedó cerca del mejor MRR de la muestra y produjo menos ventanas. El lote sube de 4 a 16 para construir el índice completo. Los textos con ambigüedades sin resolver quedan fuera.

In [ ]:
proceso = subprocess.run(
    [sys.executable, "scripts/indexar.py", "--config", "configs/indice.json"],
    cwd=raiz, capture_output=True, text=True, encoding="utf-8"
)
print(proceso.stdout)
if proceso.returncode:
    print(proceso.stderr[-6000:])
proceso.check_returncode()

In [ ]:
manifiesto = json.loads((raiz / config["salida"] / "manifest.json").read_text(encoding="utf-8"))
pd.DataFrame([{k: manifiesto[k] for k in ["unidades", "ventanas", "dimensiones", "segundos", "ram_rss_mib", "vram_pico_mib"]}])

## Reconstrucción

Se vuelve a construir en otra carpeta. Deben coincidir los hashes del corpus, la configuración y el índice.

In [ ]:
repeticion = subprocess.run(
    [sys.executable, "scripts/indexar.py", "--config", "configs/indice.json", "--salida", "data/index/repeticion"],
    cwd=raiz, capture_output=True, text=True, encoding="utf-8"
)
if repeticion.returncode:
    print(repeticion.stderr[-6000:])
repeticion.check_returncode()
segundo = json.loads((raiz / "data/index/repeticion/manifest.json").read_text(encoding="utf-8"))
comparacion = pd.DataFrame([{"hash": k, "coincide": manifiesto[k] == segundo[k]}
                           for k in ["sha256_corpus", "sha256_configuracion", "sha256_indice"]])
print(comparacion.to_string(index=False))
assert comparacion.coincide.all()
(raiz / "reports/c_reconstruccion.json").write_text(json.dumps({
    "primera": manifiesto, "segunda": segundo, "coinciden": bool(comparacion.coincide.all())
}, ensure_ascii=False, indent=2), encoding="utf-8")

## Recuperación sobre el índice completo

Se repiten las 54 sondas del experimento sobre todas las unidades admitidas. Se comparan búsqueda densa e híbrida. Cada método se ejecuta dos veces para revisar que el orden recuperado sea el mismo.

In [ ]:
from datetime import datetime, timezone

experimento = json.loads((raiz / "reports/f_encoders_ultima.json").read_text(encoding="utf-8"))
sondas = str(Path(experimento["reportes"]) / "sondas.jsonl")
salida_sondas = "reports/recuperacion_completa/" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")
proceso = subprocess.run(
    [sys.executable, "scripts/verificar_recuperacion.py", "--indice", config["salida"],
     "--sondas", sondas, "--salida", salida_sondas],
    cwd=raiz, capture_output=True, text=True, encoding="utf-8"
)
if proceso.returncode:
    print(proceso.stderr[-6000:])
proceso.check_returncode()
resumen_sondas = json.loads((raiz / salida_sondas / "resumen.json").read_text(encoding="utf-8"))
(raiz / "reports/d_completa_ultima.json").write_text(
    json.dumps({"ruta": salida_sondas}, indent=2), encoding="utf-8"
)
pd.DataFrame(resumen_sondas["resultados"])

In [ ]:
pd.read_csv(raiz / salida_sondas / "articulos_24_42.csv")[[
    "metodo", "tipo_sonda", "doc_id", "articulo", "puesto", "offsets_correctos"
]]

## Resultados y decisiones

Pendientes de ejecución. Todavía no se han comparado los hashes de dos construcciones 